In [28]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import torchmetrics
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Processing Data

In [ ]:
df = pd.read_csv("EngineFaultDB_Final.csv")
df.head()   
X = df.drop(['Fault'], axis=1).values
y = df['Fault'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
C0 = [1,1,1,1,1,1,1,1,1,1,1,1,1,1]
C1 = [1,1,1,0,0,0,1,1,1,1,1,1,1,1]

In [ ]:
def create_dataloaders(X_train, X_test, y_train, y_test, train_config, test_config, batch_size=64, use_mask=True):


    X_train = X_train.copy()
    X_test = X_test.copy()

    train_config = np.asarray(train_config, dtype=bool)
    test_config = np.asarray(test_config, dtype=bool)

    X_train[:, ~train_config] = 0.0
    X_test[:, ~test_config] = 0.0

    train_mask = np.tile(
        train_config.astype(np.float32),
        (len(X_train), 1)
    )

    test_mask = np.tile(
        test_config.astype(np.float32),
        (len(X_test), 1)
    )

    # Optionally give the model explicit information
    # about which features are available
    if use_mask:
        X_train = np.concatenate([X_train, train_mask], axis=1)
        X_test = np.concatenate([X_test, test_mask], axis=1)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)

    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test = torch.tensor(y_test, dtype=torch.long)

    # Create datasets
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    # Create loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, test_loader

In [ ]:
train_loader, test_loader = create_dataloaders(X_train, X_test, y_train, y_test, C0, C1, batch_size=64, use_mask=True)

(torch.Size([64, 14]), torch.Size([64]))

## Experiment 0: Logistic Regression

In [ ]:
INPUT_DIM = X_train.shape[1] + len(C0)  # Adjust for mask
OUTPUT_DIM = len(np.unique(y_train))
BATCH_SIZE = 64
LEARNING_RATE=0.01
EPOCHS=10

class LogisticRegressionPytorch(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LogisticRegressionPytorch, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return self.linear(x)

model_0 = LogisticRegressionPytorch(INPUT_DIM, OUTPUT_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_0.parameters(), lr=LEARNING_RATE)

def train_logistic_regression(train_loader, test_loader, model, criterion, optimizer, EPOCHS):
    for epochs in range(EPOCHS):
        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch_idx, (data, target) in enumerate(train_loader):
                data = data.to(device)
                target = target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

        # Evaluation
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for data, target in test_loader:
                data = data.to(device)
                target.to(device)
                output = model(data)
                preds = torch.argmax(output, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(target.cpu().numpy())
        accuracy = accuracy_score(all_targets, all_preds)
        multiclass_f1 = torchmetrics.classification.MulticlassF1Score(4, average='macro')
        f1_score = multiclass_f1(torch.tensor(all_preds), torch.tensor(all_targets)).item()
        print(f'Epoch {epochs+1}/{EPOCHS}, Accuracy: {accuracy:.4f}, F1-Score: {f1_score:.4f}')  

In [ ]:
train_loader, test_loader = create_dataloaders(X_train, X_test, y_train, y_test, C0, C0) 
train_logistic_regression(train_loader, test_loader, model_0, criterion, optimizer, EPOCHS)

Epoch 1/10, Accuracy: 0.1927, F1-Score: 0.1048
Epoch 2/10, Accuracy: 0.3895, F1-Score: 0.2552
Epoch 3/10, Accuracy: 0.2821, F1-Score: 0.1791
Epoch 4/10, Accuracy: 0.3471, F1-Score: 0.2514
Epoch 5/10, Accuracy: 0.2838, F1-Score: 0.1833
Epoch 6/10, Accuracy: 0.2880, F1-Score: 0.1404
Epoch 7/10, Accuracy: 0.3558, F1-Score: 0.2387
Epoch 8/10, Accuracy: 0.2903, F1-Score: 0.2152
Epoch 9/10, Accuracy: 0.2901, F1-Score: 0.1523
Epoch 10/10, Accuracy: 0.2829, F1-Score: 0.1759


In [ ]:
train_loader, test_loader = create_dataloaders(X_train, X_test, y_train, y_test, C0, C1) 
train_logistic_regression(train_loader, test_loader, model_0, criterion, optimizer, EPOCHS)

Epoch 1/10, Accuracy: 0.2130, F1-Score: 0.1174
Epoch 2/10, Accuracy: 0.2132, F1-Score: 0.1111
Epoch 3/10, Accuracy: 0.2132, F1-Score: 0.1111
Epoch 4/10, Accuracy: 0.2132, F1-Score: 0.1114
Epoch 5/10, Accuracy: 0.2098, F1-Score: 0.1225
Epoch 6/10, Accuracy: 0.2037, F1-Score: 0.1218
Epoch 7/10, Accuracy: 0.1993, F1-Score: 0.1354
Epoch 8/10, Accuracy: 0.2308, F1-Score: 0.1574
Epoch 9/10, Accuracy: 0.2013, F1-Score: 0.1218
Epoch 10/10, Accuracy: 0.2758, F1-Score: 0.1874


## Experiment 1: MLP

In [ ]:
INPUT_DIM = X_train.shape[1] + len(C0)  # Adjust for mask
OUTPUT_DIM = len(np.unique(y_train))
HIDDEN_DIM = 64
BATCH_SIZE = 64
LEARNING_RATE=0.01
EPOCHS=10

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x 

model_1 = MLPClassifier(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_1.parameters(), lr=LEARNING_RATE)

def train_mlp(train_loader, test_loader, model, criterion, optimizer, EPOCHS):
    for epochs in range(EPOCHS):
        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch_idx, (data, target) in enumerate(train_loader):
                data = data.to(device)
                target = target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

        # Evaluation
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for data, target in test_loader:
                data = data.to(device)
                target.to(device)
                output = model(data)
                preds = torch.argmax(output, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(target.cpu().numpy())
        accuracy = accuracy_score(all_targets, all_preds)
        multiclass_f1 = torchmetrics.classification.MulticlassF1Score(4, average='macro')
        f1_score = multiclass_f1(torch.tensor(all_preds), torch.tensor(all_targets)).item()
        print(f'Epoch {epochs+1}/{EPOCHS}, Accuracy: {accuracy:.4f}, F1-Score: {f1_score:.4f}')  

In [40]:
train_loader, test_loader = create_dataloaders(X_train, X_test, y_train, y_test, C0, C0)
train_mlp(train_loader, test_loader, model_1, criterion, optimizer, EPOCHS)

Epoch 1/10, Accuracy: 0.2462, F1-Score: 0.0988
Epoch 2/10, Accuracy: 0.3459, F1-Score: 0.2852
Epoch 3/10, Accuracy: 0.5732, F1-Score: 0.5494
Epoch 4/10, Accuracy: 0.6581, F1-Score: 0.6181
Epoch 5/10, Accuracy: 0.3418, F1-Score: 0.2272
Epoch 6/10, Accuracy: 0.3251, F1-Score: 0.1942
Epoch 7/10, Accuracy: 0.3251, F1-Score: 0.1942
Epoch 8/10, Accuracy: 0.2883, F1-Score: 0.1119
Epoch 9/10, Accuracy: 0.2883, F1-Score: 0.1119
Epoch 10/10, Accuracy: 0.2883, F1-Score: 0.1119


In [41]:
train_loader, test_loader = create_dataloaders(X_train, X_test, y_train, y_test, C0, C1)
train_mlp(train_loader, test_loader, model_1, criterion, optimizer, EPOCHS)

Epoch 1/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 2/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 3/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 4/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 5/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 6/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 7/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 8/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 9/10, Accuracy: 0.1959, F1-Score: 0.0819
Epoch 10/10, Accuracy: 0.1959, F1-Score: 0.0819
